# ACA Medicaid Expansion and Uninsurance Convergence Across States
ECON 630: Week 6 Assignment
Julia van Helmond

## Setup: imports

`os` reads the environment variable, `Path` handles the cache folder, `requests` calls the Census Data API,
`pandas` holds the data, and `python-dotenv` loads the API key from `.env`.

In [10]:
import os
from pathlib import Path

import pandas as pd
import requests
from dotenv import load_dotenv

## Setup: API key and cache folder

`load_dotenv()` copies the variables in `.env` into the environment, so the key is read without being
written in the notebook. The key is never printed; the cell only checks that it was found.

Each raw API response is saved to `data/raw/` (Plan §2). Later runs read those files instead of calling the API again,
which makes reruns fast and gives the same data every time.

In [11]:
load_dotenv()
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")
assert CENSUS_API_KEY, "CENSUS_API_KEY not found in .env"

RAW_DIR = Path("data/raw")
RAW_DIR.mkdir(parents=True, exist_ok=True)

## Policy table (Plan §2c)

`%run policy-dates.ipynb` loads `EXPANSION_YEAR`, the single source of truth for when each state expanded Medicaid
(KFF). It is loaded, not retyped, so the dates exist in only one place.

The Census API identifies states by FIPS code, but `EXPANSION_YEAR` uses postal codes. `FIPS_TO_POSTAL` connects the two. The asserts
check that it has 51 entries (50 states + DC) and covers exactly the same states as `EXPANSION_YEAR`.

In [12]:
%run policy-dates.ipynb

In [13]:
FIPS_TO_POSTAL = {
    "01": "AL", "02": "AK", "04": "AZ", "05": "AR", "06": "CA", "08": "CO", "09": "CT",
    "10": "DE", "11": "DC", "12": "FL", "13": "GA", "15": "HI", "16": "ID", "17": "IL",
    "18": "IN", "19": "IA", "20": "KS", "21": "KY", "22": "LA", "23": "ME", "24": "MD",
    "25": "MA", "26": "MI", "27": "MN", "28": "MS", "29": "MO", "30": "MT", "31": "NE",
    "32": "NV", "33": "NH", "34": "NJ", "35": "NM", "36": "NY", "37": "NC", "38": "ND",
    "39": "OH", "40": "OK", "41": "OR", "42": "PA", "44": "RI", "45": "SC", "46": "SD",
    "47": "TN", "48": "TX", "49": "UT", "50": "VT", "51": "VA", "53": "WA", "54": "WV",
    "55": "WI", "56": "WY",
}

assert len(FIPS_TO_POSTAL) == 51
assert set(FIPS_TO_POSTAL.values()) == set(EXPANSION_YEAR)

## Helper: call the Census API

Both data sources use the same Census API format: the first row of the JSON is the column names and the
rest are data rows. This helper sends the request, turns the response into a DataFrame, and returns `None`
when the API has no data for that request (HTTP 204), so missing years can be skipped.

On an error it reports only the status code and message. It does not show the request URL, because the URL contains the key.

In [14]:
def census_get(url, params):
    """Call the Census API and return the response as a DataFrame (None if no data)."""
    resp = requests.get(url, params={**params, "key": CENSUS_API_KEY}, timeout=60)
    if resp.status_code == 204:
        return None
    if resp.status_code != 200:
        # Leave out resp.url so the key does not show up in the output
        raise RuntimeError(f"Census API error {resp.status_code}: {resp.text[:200]}")
    rows = resp.json()
    return pd.DataFrame(rows[1:], columns=rows[0])

## ACS 1-year pull (Plan §2a)

`fetch_acs_year(year)` requests the B27010 health-insurance counts and B19013 median household income for all
states in one year. If `data/raw/acs_{year}.csv` already exists, it reads that file instead. It adds a `year` column so the years
can be stacked. Puerto Rico (FIPS 72) is removed if it appears.

The years are 2010–2019 and 2021–2022. 2020 is skipped because the Census Bureau did not release standard 2020 ACS 1-year estimates.

In [15]:
ACS_URL = "https://api.census.gov/data/{year}/acs/acs1"
ACS_VARS = [
    "B27010_001E",                                # total civilian noninstitutionalized population
    "B27010_002E", "B27010_018E", "B27010_034E",  # population, three under-65 age groups
    "B27010_017E", "B27010_033E", "B27010_050E",  # no health insurance, three under-65 age groups
    "B27010_066E",                                # no health insurance, 65 and over
    "B19013_001E",                                # median household income
]
ACS_YEARS = list(range(2010, 2020)) + [2021, 2022]


def fetch_acs_year(year):
    """Return one year of ACS state data, read from the cache when it exists."""
    cache = RAW_DIR / f"acs_{year}.csv"
    if cache.exists():
        df = pd.read_csv(cache, dtype=str)
    else:
        params = {"get": ",".join(["NAME"] + ACS_VARS), "for": "state:*"}
        df = census_get(ACS_URL.format(year=year), params)
        df.to_csv(cache, index=False)
    df = df[df["state"] != "72"].copy()  # drop Puerto Rico if present
    df[ACS_VARS] = df[ACS_VARS].apply(pd.to_numeric)
    df["year"] = year
    return df

Loop over the 12 years and stack them into one state-year DataFrame. Then show the first rows and the
number of states in each year. Each year should have 51 (50 states + DC).

In [16]:
acs = pd.concat([fetch_acs_year(y) for y in ACS_YEARS], ignore_index=True)
acs.head()

,NAME,B27010_001E,B27010_002E,B27010_018E,B27010_034E,B27010_017E,B27010_033E,B27010_050E,B27010_066E,B19013_001E,state,year
0,Alabama,4702769,1133638,1060965,1870571,66958,310679,307036,1919,40474,01,2010
1,Alaska,689249,187984,161419,288016,22843,53131,59692,1174,64576,02,2010
2,Arizona,6310174,1629931,1442598,2362085,207967,414258,433169,9738,46789,04,2010
3,Arkansas,2867611,709374,641055,1112378,46495,223941,229191,986,38307,05,2010
4,California,36815569,9288691,9023508,14326399,832752,2888172,3032396,71593,57708,06,2010


In [17]:
acs.groupby("year").size().rename("rows")

year
2010    51
2011    51
2012    51
2013    51
2014    51
2015    51
2016    51
2017    51
2018    51
2019    51
2021    51
2022    51
Name: rows, dtype: int64

## SAHIE pull (Plan §2b, robustness check)

`fetch_sahie_year(year)` does the same for the Small Area Health Insurance Estimates. It gets the percent uninsured
(`PCTUI_PT`) for adults ages 18–64 (`AGECAT=1`) with income at or below 138% of poverty (`IPRCAT=3`). That is the group
Medicaid expansion targets. It also sets `SEXCAT=0` and `RACECAT=0` (all sexes, all races) so the API returns one row per state
instead of one row per demographic cell.

The plan says not to assume every year is available. A year with no data returns `None`, gets skipped, and the
cell after the loop prints the years that actually came back.

In [18]:
SAHIE_URL = "https://api.census.gov/data/timeseries/healthins/sahie"
SAHIE_YEARS = range(2010, 2023)


def fetch_sahie_year(year):
    """Return one year of SAHIE state data (<=138% FPL, ages 18-64), or None if unavailable."""
    cache = RAW_DIR / f"sahie_{year}.csv"
    if cache.exists():
        df = pd.read_csv(cache, dtype=str)
    else:
        params = {
            "get": "NAME,PCTUI_PT",
            "for": "state:*",
            "IPRCAT": 3,   # income <= 138% of poverty
            "AGECAT": 1,   # ages 18-64
            "SEXCAT": 0,   # both sexes
            "RACECAT": 0,  # all races
            "time": year,
        }
        df = census_get(SAHIE_URL, params)
        if df is None:
            return None
        df.to_csv(cache, index=False)
    df = df[df["state"] != "72"].copy()  # drop Puerto Rico if present
    df["PCTUI_PT"] = pd.to_numeric(df["PCTUI_PT"])
    df["year"] = year
    return df

Stack the years that returned data, print which years they were, and show the first rows and the row count per year.

In [19]:
sahie_frames = [fetch_sahie_year(y) for y in SAHIE_YEARS]
sahie = pd.concat([df for df in sahie_frames if df is not None], ignore_index=True)
print("SAHIE years returned:", sorted(sahie["year"].unique().tolist()))
sahie.head()

SAHIE years returned: [2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022]


,NAME,PCTUI_PT,IPRCAT,AGECAT,SEXCAT,RACECAT,time,state,year
0,Alabama,41.6,3,1,0,0,2010,01,2010
1,Alaska,47.7,3,1,0,0,2010,02,2010
2,Wyoming,42.7,3,1,0,0,2010,56,2010
3,Arizona,39.3,3,1,0,0,2010,04,2010
4,Arkansas,47.5,3,1,0,0,2010,05,2010


In [20]:
sahie.groupby("year").size().rename("rows")

year
2010    51
2011    51
2012    51
2013    51
2014    51
2015    51
2016    51
2017    51
2018    51
2019    51
2020    51
2021    51
2022    51
Name: rows, dtype: int64

## Cleaning (Plan §3)

Each step below ends with asserts that must pass before the next step. If one fails, the notebook stops there.

### Step 1: check the stacked ACS panel

The years were already stacked above, and `fetch_acs_year` already converted the variables from strings to numbers.
This step renames the Census `state` column (a FIPS code) to `fips` and adds the postal code as `state`, which is
what the merges and regressions use from here on.

Checks: 51 states × 12 years = 612 rows, each `(state, year)` pair appears once, and 2020 is not in the data.

In [21]:
acs = acs.rename(columns={"state": "fips"})
acs["state"] = acs["fips"].map(FIPS_TO_POSTAL)

assert len(acs) == 51 * 12, len(acs)
assert not acs.duplicated(["state", "year"]).any()
assert 2020 not in acs["year"].values

### Step 2: keep the 50 states + DC

Keep only rows whose FIPS code is in `FIPS_TO_POSTAL`, which drops any territory. Then check that every year has all 51 and
that every state in the data matches a key in `EXPANSION_YEAR`, and the reverse.

The ACS marks unavailable estimates with large negative codes (e.g. `-666666666`) instead of leaving them blank, so the
missing-value check also checks that there are no negative values.

In [22]:
acs = acs[acs["fips"].isin(FIPS_TO_POSTAL)].copy()

assert (acs.groupby("year")["state"].nunique() == 51).all()
assert set(acs["state"]) == set(EXPANSION_YEAR)
assert acs[ACS_VARS].notna().all().all()
assert (acs[ACS_VARS] >= 0).all().all(), "negative ACS missing-value codes present"

### Step 3: compute the uninsurance rates

The outcome is the share of people under 65 with no health insurance: uninsured counts in the three under-65 age groups
divided by the population in those groups, times 100. Medicare covers almost everyone 65 and over, so leaving them out
focuses on the people the ACA could affect. The all-ages rate (it adds the 65+ uninsured and uses the total population)
is kept only for reference.

Check: every rate is between 0 and 40 percent.

In [23]:
UNINSURED_U65 = ["B27010_017E", "B27010_033E", "B27010_050E"]
POP_U65 = ["B27010_002E", "B27010_018E", "B27010_034E"]

acs["uninsured_rate_under65"] = acs[UNINSURED_U65].sum(axis=1) / acs[POP_U65].sum(axis=1) * 100
acs["uninsured_rate_all"] = acs[UNINSURED_U65 + ["B27010_066E"]].sum(axis=1) / acs["B27010_001E"] * 100

rates = acs[["uninsured_rate_under65", "uninsured_rate_all"]]
assert rates.gt(0).all().all() and rates.lt(40).all().all(), rates.describe()

### Step 4: assign expansion years and groups

Each state gets its expansion year from `EXPANSION_YEAR` (missing for states that never expanded), and is placed in a group:

- **Treated**: expanded in 2014 (27 states including DC).
- **Control**: had not expanded by the end of 2022, so never expanded or expanded in 2023 or later (12 states).
- **Late**: expanded 2015–2021 (12 states). These become treated partway through the sample, so they are left out of
  the main difference-in-differences and used only in a robustness check.

Check: the group counts are 27 / 12 / 12 and add up to 51.

In [24]:
def assign_group(expansion_year):
    """Classify a state by when it expanded Medicaid."""
    if pd.isna(expansion_year) or expansion_year >= 2023:
        return "control"
    if expansion_year == 2014:
        return "treated"
    return "late"  # 2015-2021


acs["expansion_year"] = acs["state"].map(EXPANSION_YEAR).astype("Int64")
acs["group"] = acs["expansion_year"].map(assign_group)
acs["treated"] = (acs["group"] == "treated").astype(int)

group_counts = acs.drop_duplicates("state")["group"].value_counts()
print(group_counts.to_string())
assert group_counts.to_dict() == {"treated": 27, "control": 12, "late": 12}
assert group_counts.sum() == 51

group
treated    27
control    12
late       12


### Step 5: post-period and treatment indicators

`post` is 1 from 2014 on, when the ACA's main coverage provisions started. `treated_post` (treated × post) is the
difference-in-differences term: it is 1 only for 2014 expansion states in 2014 and later.

The main DiD sample `did` keeps only the treated and control states. Check: 39 states × 12 years = 468 rows.

In [25]:
acs["post"] = (acs["year"] >= 2014).astype(int)
acs["treated_post"] = acs["treated"] * acs["post"]

did = acs[acs["group"].isin(["treated", "control"])].copy()
assert len(did) == 39 * 12, len(did)

### Step 6: add 2013 baseline variables

The convergence analysis compares states by where they started, so each state's 2013 values (the last year before
expansion) are attached to all of its rows: median household income (`median_income_2013`) and the under-65
uninsurance rate (`baseline_rate_2013`). The baselines are merged onto both the full panel and the DiD sample.

Check: no state is missing a baseline value.

In [26]:
baseline = (
    acs.loc[acs["year"] == 2013, ["state", "B19013_001E", "uninsured_rate_under65"]]
    .rename(columns={"B19013_001E": "median_income_2013",
                     "uninsured_rate_under65": "baseline_rate_2013"})
)

# validate= stops the merge if any state has more than one 2013 row
acs = acs.merge(baseline, on="state", how="left", validate="many_to_one")
did = did.merge(baseline, on="state", how="left", validate="many_to_one")

BASELINE_COLS = ["median_income_2013", "baseline_rate_2013"]
assert acs[BASELINE_COLS].notna().all().all()
assert did[BASELINE_COLS].notna().all().all()
did.head()

,NAME,B27010_001E,B27010_002E,B27010_018E,B27010_034E,B27010_017E,B27010_033E,B27010_050E,B27010_066E,B19013_001E,...,state,uninsured_rate_under65,uninsured_rate_all,expansion_year,group,treated,post,treated_post,median_income_2013,baseline_rate_2013
0,Alabama,4702769,1133638,1060965,1870571,66958,310679,307036,1919,40474,...,AL,16.842403,14.599739,<NA>,control,0,0,0,42849,15.829187
1,Arizona,6310174,1629931,1442598,2362085,207967,414258,433169,9738,46789,...,AZ,19.419852,16.879598,2014,treated,1,0,0,48510,20.114246
2,Arkansas,2867611,709374,641055,1112378,46495,223941,229191,986,38307,...,AR,20.286892,17.457493,2014,treated,1,0,0,40511,18.763804
3,California,36815569,9288691,9023508,14326399,832752,2888172,3032396,71593,57708,...,CA,20.691207,18.538116,2014,treated,1,0,0,60190,19.365798
4,Colorado,4956873,1226208,1176659,2017871,124128,337378,324581,3022,54046,...,CO,17.781805,15.919492,2014,treated,1,0,0,58823,15.956318
